# 06 클러스터링 방법

PCA 임베딩 + K-Means / HAC / GMM / DBSCAN 적용 후 **실루엣 계수**와 **Davies-Bouldin Index**로 품질을 평가합니다.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))
from utils.paths import DATA_PROCESSED, ML_CLUSTER
from utils.metrics import clustering_quality

dfw = pd.read_parquet(DATA_PROCESSED / 'df_weekly.parquet')

# 시계열 피벗: row=type|family, col=week
pivot = dfw.pivot_table(index=['type','family'], columns='yearweek', values='sales', fill_value=0)
meta = pivot.index.to_frame(index=False)
X_raw = pivot.values
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)
pca = PCA(n_components=min(10, X.shape[1], X.shape[0]-1), random_state=42)
X_emb = pca.fit_transform(X)
print('embedding shape:', X_emb.shape)



embedding shape: (165, 10)


In [2]:
def run_clustering(name, labels):
    q = clustering_quality(X_emb, labels)
    q['method'] = name
    return q

results = []
K = 4

# K-Means
km = KMeans(n_clusters=K, random_state=42, n_init=20)
labels_km = km.fit_predict(X_emb)
results.append(run_clustering('PCA+KMeans', labels_km))

# HAC
hac = AgglomerativeClustering(n_clusters=K)
labels_hac = hac.fit_predict(X_emb)
results.append(run_clustering('PCA+HAC', labels_hac))

# GMM
gmm = GaussianMixture(n_components=K, random_state=42, n_init=10)
labels_gmm = gmm.fit_predict(X_emb)
results.append(run_clustering('PCA+GMM', labels_gmm))

# DBSCAN
db = DBSCAN(eps=1.5, min_samples=3)
labels_db = db.fit_predict(X_emb)
results.append(run_clustering('PCA+DBSCAN', labels_db))

quality_df = pd.DataFrame(results)[['method','n_clusters','silhouette','davies_bouldin']]
print('=== 클러스터링 품질 (실루엣↑, DB Index↓) ===')
print(quality_df.round(4))



=== 클러스터링 품질 (실루엣↑, DB Index↓) ===
       method  n_clusters  silhouette  davies_bouldin
0  PCA+KMeans           4      0.7978          0.5674
1     PCA+HAC           4      0.8059          0.5770
2     PCA+GMM           4      0.4857          0.8681
3  PCA+DBSCAN           1         NaN             NaN


In [3]:
# 최적 조합 선택: 실루엣 최대 & DB 최소 (유효 클러스터>=2)
valid = quality_df[quality_df['n_clusters'] >= 2].copy()
valid['rank_score'] = valid['silhouette'].rank(ascending=False) + valid['davies_bouldin'].rank(ascending=True)
best_method = valid.sort_values('rank_score').iloc[0]['method']
print('권장 방법(초기):', best_method)

# GMM 라벨을 기본 ML 클러스터로 저장 (논문 PatchTST-GMM 전 단계: PCA+GMM)
out = meta.copy()
out['ML_CLUSTER'] = labels_gmm + 1
out['embedding_method'] = 'PCA'
out['clustering_method'] = 'GMM'
out.to_parquet(ML_CLUSTER, index=False)
quality_df.to_csv(DATA_PROCESSED / 'clustering_quality.csv', index=False)
print('저장:', ML_CLUSTER)
out.head()



권장 방법(초기): PCA+KMeans
저장: C:\Users\kjh\ai-retail-demandforecasting\data\processed\ml_cluster_type_family.parquet


,type,family,ML_CLUSTER,embedding_method,clustering_method
0,A,AUTOMOTIVE,1,PCA,GMM
1,A,BABY CARE,1,PCA,GMM
2,A,BEAUTY,1,PCA,GMM
3,A,BEVERAGES,3,PCA,GMM
4,A,BOOKS,1,PCA,GMM


## 분석 요약

### PCA 임베딩
- 165개 시계열 × 242주 → **PCA 10차원** 임베딩 (초기 단계, 09장에서 6종 임베딩으로 확장)

### 클러스터링 품질 비교 (K=4)
| 방법 | Silhouette ↑ | Davies-Bouldin ↓ | 평가 |
|------|-------------|------------------|------|
| **PCA+HAC** | **0.806** | 0.577 | 실루엣 최고 |
| **PCA+KMeans** | 0.798 | **0.567** | DB Index 최저, 종합 1위 |
| PCA+GMM | 0.486 | 0.868 | 분리도 낮음 |
| PCA+DBSCAN | — | — | 유효 클러스터 1개 (eps 부적합) |

### 해석
- **HAC·KMeans** 모두 실루엣 0.80 전후로 **양호한 군집 분리**
- DBSCAN은 165개 소규모 패널에서 파라미터 민감 → K=4 고정 방법이 적합
- 현재 **GMM 라벨을 ML_CLUSTER로 저장** (09장 PatchTST-GMM 전 단계 placeholder)
- 09장에서 6종 임베딩 비교 후 최적 조합으로 갱신 예정